# Bayesian Grounding Verdict Calibration

The grounding verdict is a **calibrated probability** `P(grounded)` produced by a Bayesian
logistic model (**bambi / PyMC**) over the per-layer grounding scores. This notebook shows the
whole loop: the shipped prior, fitting on labelled data, posterior uncertainty, prediction,
per-language evaluation, incremental updates, and saving/loading a profile.

Key design choice: the meaning signal is the **model/language-portable** `semantic_ratio`
(ramped), not raw cosine - so a cross-lingual true match (no word overlap) can confirm while a
topical fabrication does not. The library does all the Bayesian work; nothing is hand-rolled.

## Imports

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json
import numpy as np
import pandas as pd
import arviz as az
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from groundrails import calibration as C

print("predictors:", C.PREDICTORS)

predictors: ['exact', 'fuzzy', 'bm25_recall', 'semantic', 'voters', 'lexical_cosupport', 'entity_absent']


## 1. A synthetic multilingual labelled set

Each row is a claim's grounding feature vector plus a gold `grounded` label and a `lang`.
Real production labels (en / nb-NO / fr-FR) live elsewhere; this stand-in shows the mechanics.

In [2]:
def row(grounded, lang, exact=0, fuzzy=0.0, bm25=0.0, sem=0.0, voters=0.0, cosup=0, eabs=0.0):
    return dict(exact=float(exact), fuzzy=fuzzy, bm25_recall=bm25, semantic=sem,
                voters=voters, lexical_cosupport=float(cosup), entity_absent=eabs,
                grounded=grounded, lang=lang)

rng = np.random.default_rng(0)
def clip(x): return float(min(1.0, max(0.0, x)))
rows = []
for lang in ["en", "nb", "fr"]:
    for _ in range(3):
        rows.append(row(1, lang, exact=1, fuzzy=0.95, bm25=0.8, sem=0.6, voters=0.75, cosup=1))   # exact / lexical
        rows.append(row(1, lang, sem=clip(0.82 + 0.03*rng.standard_normal()), voters=0.25))       # cross-lingual semantic
        rows.append(row(0, lang, sem=clip(0.15 + 0.05*rng.standard_normal()), voters=0.25))       # topical fabrication
        rows.append(row(0, lang, sem=0.30, voters=0.25, eabs=1.0))                                # entity-absent fabrication
df = pd.DataFrame(rows)
print(df.shape)
df.head()

(36, 9)


,exact,fuzzy,bm25_recall,semantic,voters,lexical_cosupport,entity_absent,grounded,lang
0,1.0,0.95,0.8,0.600000,0.75,1.0,0.0,1,en
1,0.0,0.00,0.0,0.823772,0.25,0.0,0.0,1,en
2,0.0,0.00,0.0,0.143395,0.25,0.0,0.0,0,en
3,0.0,0.00,0.0,0.300000,0.25,0.0,1.0,0,en
4,1.0,0.95,0.8,0.600000,0.75,1.0,0.0,1,en


## 2. The shipped default calibrator (untrained prior)

`default_calibrator()` fits the interpretable prior against a small anchor set, so it is usable
out of the box. It already separates a cross-lingual true match from a fabrication.

In [3]:
cal0 = C.default_calibrator(draws=500, tune=500, random_seed=0)

cases = {
    "exact":        {"exact": 1, "fuzzy": 0.95, "bm25_recall": 0.8, "voters": 0.75, "lexical_cosupport": 1},
    "crosslingual": {"semantic": 0.82, "voters": 0.25},
    "fabrication":  {"semantic": 0.15, "voters": 0.25},
    "entity_absent":{"semantic": 0.30, "voters": 0.25, "entity_absent": 1.0},
}
for name, feat in cases.items():
    p, s = cal0.predict_with_uncertainty(feat)
    verdict = "CONFIRMED" if cal0.confirmed(feat) else "not"
    print(f"{name:14s} P={p[0]:.3f} +/- {s[0]:.3f}  -> {verdict}")

Modeling the probability that grounded==1


Initializing NUTS using jitter+adapt_diag...


Sequential sampling (2 chains in 1 job)


NUTS: [Intercept, exact, fuzzy, bm25_recall, semantic, voters, lexical_cosupport, entity_absent]


Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 0 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


exact          P=0.893 +/- 0.178  -> CONFIRMED
crosslingual   P=0.734 +/- 0.129  -> CONFIRMED
fabrication    P=0.025 +/- 0.029  -> not
entity_absent  P=0.003 +/- 0.008  -> not


## 3. Calibrate on the labelled data (bambi / PyMC)

`fit_calibrator` runs Bayesian logistic regression - a full posterior over the coefficients.

In [4]:
cal = C.fit_calibrator(df, draws=1000, tune=1000, random_seed=0)
az.summary(cal.idata, var_names=list(C._PRIOR.keys()), kind="stats")

Modeling the probability that grounded==1


Initializing NUTS using jitter+adapt_diag...


Sequential sampling (2 chains in 1 job)


NUTS: [Intercept, exact, fuzzy, bm25_recall, semantic, voters, lexical_cosupport, entity_absent]


Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 1 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


,mean,sd,eti89_lb,eti89_ub
Intercept,-5.5,1.3,-7.7,-3.5
exact,4.3,1.8,1.3,7.3
fuzzy,0.24,1.8,-2.6,3.1
bm25_recall,1.1,1.9,-1.9,4.1
semantic,8,1.6,5.4,11
voters,1,1.4,-1.3,3.4
lexical_cosupport,1.1,1.4,-1.1,3.4
entity_absent,-4.2,1.5,-6.6,-2


## 4. Posterior coefficients (uncertainty)

Each coefficient is a distribution, not a point estimate - the width is the model's uncertainty.

In [5]:
az.plot_forest(cal.idata, var_names=list(C._PRIOR.keys()), combined=True)
plt.tight_layout()
plt.show()

## 5. Predict with uncertainty

The posterior-predictive mean is the calibrated probability; its spread flags borderline claims.

In [6]:
for name, feat in cases.items():
    p, s = cal.predict_with_uncertainty(feat)
    print(f"{name:14s} P={p[0]:.3f} +/- {s[0]:.3f}  -> {'CONFIRMED' if cal.confirmed(feat) else 'not'}")

exact          P=0.716 +/- 0.291  -> CONFIRMED
crosslingual   P=0.761 +/- 0.119  -> CONFIRMED
fabrication    P=0.027 +/- 0.030  -> not
entity_absent  P=0.003 +/- 0.006  -> not


## 6. Evaluate: precision / recall + per language

Per-language parity matters most for the multilingual case - the portable `semantic_ratio` feature
is what holds parity across en / nb / fr.

In [7]:
print(json.dumps(C.evaluate(cal, df), indent=2))

{
  "n": 36,
  "precision": 1.0,
  "recall": 1.0,
  "f1": 1.0,
  "accuracy": 1.0,
  "tp": 18,
  "fp": 0,
  "fn": 0,
  "tn": 18,
  "by_lang": {
    "en": {
      "n": 12,
      "precision": 1.0,
      "recall": 1.0,
      "f1": 1.0,
      "accuracy": 1.0,
      "tp": 6,
      "fp": 0,
      "fn": 0,
      "tn": 6
    },
    "nb": {
      "n": 12,
      "precision": 1.0,
      "recall": 1.0,
      "f1": 1.0,
      "accuracy": 1.0,
      "tp": 6,
      "fp": 0,
      "fn": 0,
      "tn": 6
    },
    "fr": {
      "n": 12,
      "precision": 1.0,
      "recall": 1.0,
      "f1": 1.0,
      "accuracy": 1.0,
      "tp": 6,
      "fp": 0,
      "fn": 0,
      "tn": 6
    }
  }
}


## 7. Incremental update (posterior-as-prior)

New feedback does not reset the model: the previous posterior seeds the next fit, so calibration
accumulates. This is what the `calibrate update` CLI will do from grounding + LLM-eval feedback.

In [8]:
new = df.sample(8, random_state=1)
cal2 = C.update_calibrator(cal, new, draws=500, tune=500, random_seed=2)
before, after = cal.posterior_summary(), cal2.posterior_summary()
for k in ["Intercept", "semantic", "bm25_recall", "exact"]:
    print(f"{k:12s} {before[k][0]:+.2f}  ->  {after[k][0]:+.2f}")

Modeling the probability that grounded==1


Initializing NUTS using jitter+adapt_diag...


Sequential sampling (2 chains in 1 job)


NUTS: [Intercept, exact, fuzzy, bm25_recall, semantic, voters, lexical_cosupport, entity_absent]


Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 0 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


Intercept    -5.50  ->  -6.90
semantic     +8.00  ->  +8.80
bm25_recall  +1.10  ->  +0.91
exact        +4.30  ->  +4.20


## 8. Save / load a profile

The fitted posterior persists as one JSON (no HDF5 backend needed) and reloads for prediction.

In [9]:
import tempfile, os.path as op
path = op.join(tempfile.mkdtemp(), "calibrator.json")
cal.save(path)
cal_loaded = C.CalibratedVerdict.load(path)
print("profile bytes:", op.getsize(path))
print("reloaded P(crosslingual):", round(float(cal_loaded.predict_proba(cases["crosslingual"])[0]), 3))

profile bytes: 319115
reloaded P(crosslingual): 0.761
